# Process Multiple Long Strips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,"Z, pixel size",Angle,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),File size (MB),Total size (GB),Unnamed: 22
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,2025-03-27,GHL_pyapp_20250327T1450,oven aging test day 2,strip 14,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1.0,2025-04-14,GHL_pyapp_20250414T1541,0.3-0.324 mg/mm,0.302 02-13 10,0.302,1-sided magnetic,7.0,9.0,1.5,...,2.17,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.0,8.1,Accidentually deleted
7,2.0,2025-04-14,GHL_pyapp_20250414T1610,0.3-0.324 mg/mm,0.308 02-26 11,0.308,1-sided magnetic,7.0,9.0,1.5,...,2.17,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.0,8.1,NaN
8,3.0,2025-04-14,GHL_pyapp_20250414T1618,0.3-0.324 mg/mm,0.302 02-13 10,0.302,1-sided magnetic,7.0,9.0,1.5,...,2.17,NaN,105.1,-60.0,165.1,"Medium, 48kHz",2.0,425.0,8.1,Repeat test 1
9,4.0,2025-04-14,GHL_pyapp_20250414T1636,0.3-0.324 mg/mm,ES0331 #26 0.320,0.320,1-sided magnetic,7.0,9.0,1.5,...,2.17,NaN,104.8,-60.5,165.3,"Medium, 48kHz",2.0,425.0,8.1,NaN


In [4]:
#%% Filter The Tests To Process
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     dftests['Test date']=='2025-05-05',
# ]
dfmasks = [
    dftests['Test date']=='2025-05-08',
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,"Z, pixel size",Angle,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),File size (MB),Total size (GB),Unnamed: 22
22,17.0,2025-05-08,GHL_pyapp_20250508T1112,0.325-0.35,ES0408#17 0.327,0.327,Vacuum,9.0,3.5,1.5,...,2.17,0.0,307.5,142.5,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
23,18.0,2025-05-08,GHL_pyapp_20250508T1128,0.325-0.35,ES0409#8 0.333,0.333,Vacuum,9.0,3.5,1.5,...,2.17,0.0,307.5,142.5,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
24,19.0,2025-05-08,GHL_pyapp_20250508T1134,0.325-0.35,ES0410#18 0.333,0.333,Vacuum,9.0,3.5,1.5,...,2.17,0.0,307.5,142.5,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
25,20.0,2025-05-08,GHL_pyapp_20250508T1143,0.375-0.4,ES0401#21 0.391,0.391,Vacuum,9.0,3.5,1.5,...,2.17,0.0,307.5,142.5,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
26,21.0,2025-05-08,GHL_pyapp_20250508T1151,0.375-0.4,ES0403#33 0.392,0.392,Vacuum,9.0,3.5,1.5,...,2.17,0.0,308.0,143.0,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
27,22.0,2025-05-08,GHL_pyapp_20250508T1158,0.375-0.4,ES0410#24 0.380,0.380,Vacuum,9.0,3.5,1.5,...,2.17,0.0,308.0,143.0,165.0,"Medium, 48kHz",2.0,213.0,4.1,NUM_FOVS: remove +1
28,23.0,2025-05-08,GHL_pyapp_20250508T1539,0.325-0.349mg/mm lrg bag,ES0410#20 0.338,0.338,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simon (my data fields from today to be populat...
29,24.0,2025-05-08,GHL_pyapp_20250508T1545,0.325-0.349mg/mm lrg bag,ES0408#3 0.334,0.334,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simon
30,25.0,2025-05-08,GHL_pyapp_20250508T1551,0.325-0.349mg/mm lrg bag,ES0408#13 0.341,0.341,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simon
31,26.0,2025-05-08,GHL_pyapp_20250508T1555,0.325-0.349mg/mm lrg bag,ES0408#30 0.349,0.349,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Simon


In [5]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


STUDY: GHL_pyapp_20250508T1112
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1128
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1134
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1143
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250508T1151
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

In [6]:
# Master Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            octstudy.vdvol = vdvol;
        else:
            # Load OCT Data for this study
            octstudy.load_all_octs();


            # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

            # RGB Camera Images - Write them out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

            # RGB Camera Images - Make a montage and write out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
            #break;


            # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
            octstudy.folder_study_processed.mkdir(exist_ok=True);
            fname = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
            if(fname.exists()):
                print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                print(fname);
            else:
                print(f'Generating merged and rescaled .vtk volume');
                # Generate merged and rescaled volume
                vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
                octstudy.vdvol = vdvol;
            
                # save this byte-adjusted volume
                vdvol.dataset.save(fname);
            
            # Unload OCT Data for this study (we will still keep the vdvol)
            octstudy.unload_all_octdata();

        # --Detailed Image Processing--
        del octstudy.vdvol;

<OCT_Study_Folder Object>
GHL_pyapp_20250508T1112 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
Loading GHL_pyapp_20250508T1112_STACKED_RESCALED_30to60_uint8.vtk
<OCT_Study_Folder Object>
GHL_pyapp_20250508T1128 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
Loading GHL_pyapp_20250508T1128_STACKED_RESCALED_30to60_uint8.vtk
<OCT_Study_Folder Object>
GHL_pyapp_20250508T1134 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True

In [10]:
del vdvol

# Call Notebooks - Processing A to D

In [11]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    pp.pprint(octstudy.resultsCheck())
    doWeRunTheNotebook = any([v is False for k,v in octstudy.resultsCheck().items()])
    print('Run?',doWeRunTheNotebook)
    if(doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

if(len(octstudies_to_run)>4):
    octstudies_to_run=octstudies_to_run[0:4];
print('only first 4');

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

~~~~~~~
GHL_pyapp_20250508T1112
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250508T1128
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250508T1134
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250508T1143
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250508T1151
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False,
    'figoutmp4_stepB': False}
Run? True
~~~~~~~
GHL_pyapp_20250508T1158
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
  

In [12]:
import papermill

#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\nb_test_nbparams.ipynb";
#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250429_process_a_longstrip.ipynb";
nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250505_process_a_longstrip.ipynb";

parameters = papermill.inspect_notebook(nbpath)
pp.pprint(parameters);

{   'codename': {   'default': "'20250505';",
                    'help': '',
                    'inferred_type_name': 'None',
                    'name': 'codename'},
    'folder_figure_temp': {   'default': "r'D:\\\\TEMP\\\\OCTtmp\\\\';",
                              'help': '',
                              'inferred_type_name': 'None',
                              'name': 'folder_figure_temp'},
    'folder_octexport_root': {   'default': "Path(r'\\\\file.corp.ghlabs.org\\Shared\\Projects\\NAATOS\\V1\\NAATOS_OCT_WORK\\OCTExport')",
                                 'help': '',
                                 'inferred_type_name': 'None',
                                 'name': 'folder_octexport_root'},
    'oct_scalar_max': {   'default': '60;',
                          'help': '',
                          'inferred_type_name': 'None',
                          'name': 'oct_scalar_max'},
    'oct_scalar_min': {   'default': '30;',
                          'help': '',
        

# Call notebooks to process (serially)

In [ ]:
for octstudy in octstudies_to_run:
    print(f'Now launching the processing notebook on {octstudy.name}');
    
    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'C:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(codename = Path(nbpath).name,study_name=octstudy.name)
    )

# Call notebooks to process (Multiple-strips in parallel, concurrent futures)

In [15]:
nbpath_out = Path(r'D:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
nbpath_out.parent

WindowsPath('D:/TEMP/OCTtmp')

In [16]:
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):
    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'D:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(
            folder_octexport_root=folder_octexport_root.as_posix(),
            codename = Path(nbpath).name,
            study_name=octstudy.name,
            folder_figure_temp=nbpath_out.parent.as_posix()
        )
    )
    # print('start',octstudy.name);
    # time.sleep(1)
    # print('stop',octstudy.name)

# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result());

# Example usage
num_processes = 8;  # Number of parallel processes
run_in_parallel(num_processes);

D:\TEMP\OCTtmp\octproc_20250505_process_a_longstrip_OUTGHL_pyapp_20250508T1112.ipynb
D:\TEMP\OCTtmp\octproc_20250505_process_a_longstrip_OUTGHL_pyapp_20250508T1128.ipynb
D:\TEMP\OCTtmp\octproc_20250505_process_a_longstrip_OUTGHL_pyapp_20250508T1134.ipynb
D:\TEMP\OCTtmp\octproc_20250505_process_a_longstrip_OUTGHL_pyapp_20250508T1143.ipynb


Executing:   0%|          | 0/54 [00:00<?, ?cell/s]

Executing:   0%|          | 0/54 [00:00<?, ?cell/s]

Executing:   0%|          | 0/54 [00:00<?, ?cell/s]

Executing:   0%|          | 0/54 [00:00<?, ?cell/s]

PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [29]":
---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
Cell In[29], line 21
     18 output_filename = folder_figure_temp/'{:s}{:04d}.jpg'.format(filebase,count);
     20 if( (overwrite and os.path.exists(output_filename)) or (not os.path.exists(output_filename))):
---> 21     data,fig = process_a_crosssection_sliceA(seg_sliced,slab_thickness,slicecenter,mkplot=mkplot);
     23     if mkplot:
     24         fig.savefig(output_filename);

Cell In[27], line 367, in process_a_crosssection_sliceA(seg, slab_thickness_px, slice_position_along_strip_px, mkplot)
    364         sliceinfo['wax_roi_valve_bottom'] = pksB[0][pksqualifying][selected_peak_in_B].item();
    365     else:
    366         # pick last peak from pksB2
--> 367         sliceinfo['wax_roi_valv

In [ ]:
import concurrent.futures
import time

def task(n):
    time.sleep(1)
    return n * n

# Using ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(task, i) for i in range(10)]
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

# # Using ProcessPoolExecutor
# with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
#     results = executor.map(task, range(10))
#     for result in results:
#         print(result)

# Call Notebooks - Processing E

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    rescheck = octstudy.resultsCheck();
    pp.pprint(rescheck);
    doWeRunTheNotebook = not any([v is False for k,v in rescheck.items()])
    doWeRunTheNotebook = doWeRunTheNotebook and ('/dfstepE' in rescheck['along_strip_data_extracted']);
    print('Run?',not doWeRunTheNotebook)
    if(not doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

In [ ]:
# Prepare notebook to run
import papermill

#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\nb_test_nbparams.ipynb";
#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250429_process_a_longstrip.ipynb";
nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250507_process_E_longstripmetric.ipynb";

parameters = papermill.inspect_notebook(nbpath)
pp.pprint(parameters);

In [ ]:
# Call notebooks in parallel
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):
    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'C:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT_{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(codename = Path(nbpath).name,study_name=octstudy.name)
    )
    # print('start',octstudy.name);
    # time.sleep(1)
    # print('stop',octstudy.name)

# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

# Example usage
num_processes = 4;  # Number of parallel processes
run_in_parallel(num_processes);

# Processing Step 0 - Replace VTK scalars with scaled bytes

In [ ]:
#del vdvol;
#del scalars_rescaled_as_int

# Develop VTK Range Rescaling and integer

In [ ]:
def doVtkImageMinMaxScaling(vtkimagedata):
    # restrict range and re-scale, cast to integer (will reduce memory by 4x)
    oct_scalar_min = 30;
    oct_scalar_max = 60;

    import vtk

    # Create VTK Pipeline Connections
    # 1. Thresholding Hi
    alg_thresholder = vtk.vtkImageThreshold();
    alg_thresholder.SetInputData(vtkimagedata)
    alg_thresholder.ThresholdByUpper(oct_scalar_min);
    alg_thresholder.SetOutValue(oct_scalar_min);
    alg_thresholder.ReplaceInOff();
    alg_thresholder.ReplaceOutOn();

    # 2. Thresholding Lo
    alg_thresholder2 = vtk.vtkImageThreshold();
    alg_thresholder2.SetInputConnection(alg_thresholder.GetOutputPort());
    alg_thresholder2.ThresholdByLower(oct_scalar_max);
    alg_thresholder2.SetOutValue(oct_scalar_max);
    alg_thresholder2.ReplaceInOff();
    alg_thresholder2.ReplaceOutOn();

    # 3. Subtract Minimum Value
    alg_math1 = vtk.vtkImageMathematics();
    alg_math1.SetInputConnection(alg_thresholder2.GetOutputPort());
    alg_math1.SetConstantC(-oct_scalar_min);
    alg_math1.SetOperationToAddConstant();

    # 4. Rescale to 0 to 255
    alg_math2 = vtk.vtkImageMathematics();
    alg_math2.SetInputConnection(alg_math1.GetOutputPort());
    alg_math2.SetConstantK(255.0/oct_scalar_min);
    alg_math2.SetOperationToMultiplyByK();
    #cast to unit8_t byte
    #alg_math2.SetOutputS();

    # 5. Cast to Uint8
    alg_caster = vtk.vtkImageCast();
    alg_caster.SetInputConnection(alg_math2.GetOutputPort());
    alg_caster.SetOutputScalarTypeToUnsignedChar();


    # execute the vtk pipline, wrap with pyvista, and return
    alg_final = alg_caster;
    alg_final.Update();

    final = pv.wrap(alg_final.GetOutput())
    return final;
mergedvol_rescaled = doVtkImageMinMaxScaling(mergedvol);

In [ ]:
np.max(mergedvol.active_scalars)

In [ ]:
print(np.min(mergedvol_rescaled.active_scalars),np.max(mergedvol_rescaled.active_scalars))

In [ ]:
mergedvol_rescaled['OCTintensity']

In [ ]:
del mergedvol,mergedvol_rescaled,alg_thresholder

In [ ]:
del alg_final